In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [2]:
movies = pd.read_csv("IMDB Dataset.csv")

## Data cleaning + Tokenization

In [3]:
movies.duplicated().sum()

np.int64(418)

In [4]:
movies = movies.drop_duplicates().reset_index(drop=True)

In [5]:
stop_words = set(stopwords.words('english'))

In [6]:
ps = PorterStemmer()
corpus = []

In [7]:
for i in range(len(movies)): # for each comment : 
    review = re.sub('<.*?>', ' ', movies['review'][i])  # delete html parts
    review = re.sub('[^a-zA-Z]', ' ', review)  # keep only letters 
    review = review.lower() # keep only lower caracteres
    review = review.split() # tokenization
    review = [ps.stem(word) for word in review if word not in stop_words]  # keep only important words, and reduce each word to its root form
    review = ' '.join(review) # convert list of words into a single string
    corpus.append(review) # add each review to the corpus

## Vectorization

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
tfidf = TfidfVectorizer(max_features=5000) # use TfidfVectorizer method 
X = tfidf.fit_transform(corpus) # transform each review into a numerical matrix 50000 x 5000 (number of remaining words)
y = movies['sentiment'] # output vector (positive or negative)

## Training

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Logistic Regression

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model_logistic_regression = LogisticRegression() 
model_logistic_regression.fit(X_train, y_train) # train model using logistic regression
y_pred = model_logistic_regression.predict(X_test) # prediction
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}") # check accuracy
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive'])) # create a performance report 

Accuracy : 0.8859
              precision    recall  f1-score   support

    Negative       0.90      0.87      0.88      4939
    Positive       0.87      0.90      0.89      4978

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



### Linear SVC

In [12]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
param_grid = {'C': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]} 
grid = GridSearchCV(LinearSVC(max_iter=2000), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)
print(f"Meilleur C : {grid.best_params_}")                                                   
print(f"accuracy : {accuracy_score(y_test, grid.predict(X_test)):.4f}")                      
print(classification_report(y_test, grid.predict(X_test), target_names=['Negative', 'Positive']))  

Meilleur C : {'C': 0.1}
accuracy : 0.8870
              precision    recall  f1-score   support

    Negative       0.90      0.87      0.88      4939
    Positive       0.87      0.91      0.89      4978

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



In [13]:
model_linear_svc = LinearSVC()
model_linear_svc.fit(X_train, y_train)
y_pred = model_linear_svc.predict(X_test)
print(f"accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

accuracy : 0.8786
              precision    recall  f1-score   support

    Negative       0.89      0.86      0.88      4939
    Positive       0.87      0.89      0.88      4978

    accuracy                           0.88      9917
   macro avg       0.88      0.88      0.88      9917
weighted avg       0.88      0.88      0.88      9917



### Naives Bayes

In [14]:
from sklearn.naive_bayes import MultinomialNB

In [15]:
model_naives_bayes = MultinomialNB()
model_naives_bayes.fit(X_train, y_train)
y_pred = model_naives_bayes.predict(X_test)
print(f"Accuracy : , {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

Accuracy : , 0.8492
              precision    recall  f1-score   support

    Negative       0.86      0.84      0.85      4939
    Positive       0.84      0.86      0.85      4978

    accuracy                           0.85      9917
   macro avg       0.85      0.85      0.85      9917
weighted avg       0.85      0.85      0.85      9917

